# Shapley vs Banzhaf on a toy majority game

**Goal:** Show that identical coalition samples yield different attributions depending on the **estimand** (Shapley vs Banzhaf).

**Prerequisites:** `nlp-shap` 0.1.1+, Python 3.12.

**Theory:** See the [estimands theory page](https://pawlo77.github.io/nlp-shap/theory/estimands.html) in the deployed docs.

**Expected output:** Shapley values ≈ `2/6` per player; Banzhaf indices = `0.5` per player on a 3-player majority game.

In [1]:
from itertools import product

from nlp_shap import (
    BanzhafAggregator,
    Estimand,
    ExplainResult,
    RunManifest,
    ShapleyAggregator,
    parse_manifest,
)

## Define the characteristic function

Three players; payoff is 1 when at least two are present (majority coalition), else 0. This game is **not additive**, so Shapley and Banzhaf disagree.

In [2]:
NUM_PLAYERS = 3

masks = [tuple(bits) for bits in product([False, True], repeat=NUM_PLAYERS)]


def majority_payoff(mask: tuple[bool, ...]) -> float:
    return 1.0 if sum(mask) >= 2 else 0.0


payoffs = [majority_payoff(mask) for mask in masks]
len(masks), payoffs[:4]

(8, [0.0, 0.0, 0.0, 1.0])

In [3]:
shapley = ShapleyAggregator().aggregate(masks, payoffs)
banzhaf = BanzhafAggregator().aggregate(masks, payoffs)

print("Shapley:", shapley)
print("Banzhaf:", banzhaf)
assert shapley != banzhaf

Shapley: [0.3333333333333333, 0.3333333333333333, 0.3333333333333333]
Banzhaf: [0.5, 0.5, 0.5]


## Label results for archives

Every attribution output records its estimand so downstream analysis cannot mislabel Banzhaf as Shapley.

In [4]:
result = ExplainResult(estimand=Estimand.SHAPLEY, values=tuple(shapley))
manifest = RunManifest(estimand=result.estimand, run_id="toy-majority")
restored = parse_manifest(manifest.to_dict())

result.estimand, restored.estimand, restored.run_id

(<Estimand.SHAPLEY: 'shapley'>, <Estimand.SHAPLEY: 'shapley'>, 'toy-majority')